# 04 - Results and Evaluation
Evaluate model outputs and inspect arbitrage signals with explainability diagnostics.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for package in ["yfinance", "shap", "plotly"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config import (
    ARBITRAGE_WINDOW,
    DEFAULT_HORIZON,
    DEFAULT_WINDOW,
    PAIR_CONFIGS,
    VOLATILITY_FILTER_QUANTILE,
    ZSCORE_ENTRY,
    ZSCORE_EXIT,
)
from src.anomaly_detector import ResidualAnomalyDetector
from src.arbitrage_detector import ArbitrageDetector
from src.arbitrage_signal import ArbitrageSignalGenerator
from src.data_loader import MarketDataLoader
from src.explainability import TrackingErrorExplainer
from src.features import FeatureEngineer
from src.models import TrackingErrorModel
from src.real_time_predictor import RealTimeTrackingErrorPredictor
from src.utils import time_split

# Load data and build features (daily, 2-year window)
loader   = MarketDataLoader()
panel    = loader.fetch_universe(PAIR_CONFIGS, period="2y", interval="1d")
features = FeatureEngineer(rolling_window=DEFAULT_WINDOW, horizon=DEFAULT_HORIZON).transform_universe(panel)

# Load the persisted model trained in notebook 03
model_path = project_root / "artifacts" / "te_model.joblib"
if not model_path.exists():
    raise FileNotFoundError(
        f"Model artifact not found at {model_path}. Run notebook 03 training save cell first."
    )
model = TrackingErrorModel.load(model_path)

# Score the full feature set (used in subsequent cells)
features_clean = features.dropna()
feat_cols      = model.numeric_columns + model.categorical_columns
features_clean = features_clean.dropna(subset=feat_cols)
x              = features_clean[feat_cols]
pred           = model.predict(x)

print(f"Model loaded from  : {model_path}")
print(f"Scored observations: {len(pred):,}")
print(f"Pairs              : {features_clean['pair'].unique().tolist()}")
pred[:5]

array([0.00037953, 0.00041809, 0.00068396, 0.00069541, 0.00042138])

In [ ]:
# --- Legacy spread-based signal snapshot (ArbitrageDetector) ------------------
# This module provides the statistical z-score dislocation view used in monitoring.
detector     = ArbitrageDetector(
    window=ARBITRAGE_WINDOW,
    zscore_entry=ZSCORE_ENTRY,
    zscore_exit=ZSCORE_EXIT,
    volatility_filter_quantile=VOLATILITY_FILTER_QUANTILE,
)

pair_name    = features_clean["pair"].iloc[-1]
pair_panel   = panel[panel["pair"] == pair_name].sort_index()
signal_snap  = detector.latest_signal(pair_panel)

print(f"Pair            : {pair_name}")
print(f"Signal          : {signal_snap['signal']}")
print(f"Spread z-score  : {signal_snap['spread_zscore']:.4f}")
print(f"Spread vol      : {signal_snap['spread_vol']:.6f}")
print(f"Mean-rev half-life (days): {signal_snap['half_life']:.1f}")

{'signal': 'WATCH',
 'spread_zscore': 1.3292874727656008,
 'spread_vol': 0.00036324831159242085,
 'half_life': 71.12342430422771}

In [ ]:
# --- Per-pair arbitrage signals (ArbitrageSignalGenerator) --------------------
# This is the production signal module used by the dashboard.
# Combines predicted TE direction, persistence of recent realized TE, and
# ETF dollar-volume liquidity into a single confidence-gated signal.
signal_gen = ArbitrageSignalGenerator(
    confidence_threshold=0.70,
    entry_tracking_error=0.0005,
    max_notional=1_500_000.0,
    min_notional=100_000.0,
    transaction_cost_bps=3.0,
    slippage_bps=2.0,
    persistence_window=max(6, ARBITRAGE_WINDOW // 10),
    liquidity_window=12,
)

# Build latest_predictions snapshot required by generate_universe_signals().
latest_rows = []
for p_name, p_df in features_clean.groupby("pair"):
    latest_p = p_df.tail(1)
    x_p      = latest_p[feat_cols]
    pte      = float(model.predict(x_p)[0])
    latest_rows.append({"pair": p_name, "predicted_tracking_error": pte})

latest_predictions = pd.DataFrame(latest_rows)
universe_signals   = signal_gen.generate_universe_signals(
    intraday_panel=panel,
    prediction_snapshot=latest_predictions,
)
universe_signals[[
    "pair", "action", "confidence",
    "recommended_shares", "notional",
    "estimated_profit", "estimated_profit_bps", "reason",
]]

,feature,mean_abs_shap
38,num__te_roll_std_20,0.001874
42,num__spread_std_20,0.000144
48,num__rolling_beta,0.000127
54,cat__pair_FEZ_^STOXX50E,0.000115
36,num__spread_mean_10,0.000088
33,num__te_roll_std_10,0.000072
39,num__etf_vol_20,0.000029
47,num__spread_std_60,0.000022
40,num__benchmark_vol_20,0.000020
32,num__spread_std_5,0.000019


In [ ]:
# --- Real-time / intraday prediction series (RealTimeTrackingErrorPredictor) --
# The predictor serves intraday forecasts with prediction intervals.  Here we
# run it on the daily panel as a demonstration of the interface.
rt_predictor = RealTimeTrackingErrorPredictor(
    model=model,
    rolling_window=DEFAULT_WINDOW,
    horizon=DEFAULT_HORIZON,
)
rt_features       = rt_predictor.build_feature_panel(panel)
latest_rt_preds   = rt_predictor.predict_latest(rt_features, confidence_level=0.95)
prediction_series = rt_predictor.build_prediction_series(rt_features, confidence_level=0.95)

print("Latest predictions with 95% intervals:")
print(latest_rt_preds.to_string(index=False))

# Visualize prediction time-series for the first pair
first_pair = latest_rt_preds["pair"].iloc[0]
pair_series = prediction_series[prediction_series["pair"] == first_pair].sort_values("timestamp")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pair_series["timestamp"],
    y=pair_series["predicted_tracking_error"] * 100,
    mode="lines", name="Predicted TE", line=dict(color="#103a6f", width=2),
))
fig.add_trace(go.Scatter(
    x=pair_series["timestamp"], y=pair_series["ci_upper"] * 100,
    mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip",
))
fig.add_trace(go.Scatter(
    x=pair_series["timestamp"], y=pair_series["ci_lower"] * 100,
    mode="lines", line=dict(width=0),
    fill="tonexty", fillcolor="rgba(16,58,111,0.12)",
    name="95% CI",
))
fig.update_layout(
    title=f"{first_pair}: Predicted Tracking Error with 95% Confidence Interval",
    xaxis_title="Date", yaxis_title="Predicted TE (%)",
    template="plotly_white", height=430,
)
fig.show()

In [ ]:
# --- Per-observation SHAP waterfall (TrackingErrorExplainer) ------------------
# TrackingErrorExplainer wraps TreeExplainer around the trained pipeline and
# returns a ranked DataFrame of: feature, shap_value, direction, prediction.
explainer = TrackingErrorExplainer(model)

# Use the latest observation from the most active pair for the waterfall.
pair_for_shap = first_pair
shap_features = rt_features[rt_features["pair"] == pair_for_shap].dropna(subset=feat_cols)
latest_obs    = shap_features.tail(1)[feat_cols]

if latest_obs.empty:
    print(f"No complete observation available for {pair_for_shap}. Skipping SHAP waterfall.")
else:
    shap_df = explainer.explain_observation(latest_obs)
    top_shap = shap_df.head(10).sort_values("shap_value", key=abs, ascending=False)

    print(f"Pair: {pair_for_shap}  |  Prediction: {top_shap['prediction'].iloc[0]*100:.4f}%")
    print(top_shap[["feature", "shap_value", "direction"]].to_string(index=False))

    colors = ["#cf2e2e" if v > 0 else "#1d8a4c" for v in top_shap["shap_value"]]
    fig2 = go.Figure(go.Bar(
        x=top_shap["feature"],
        y=top_shap["shap_value"] * 100,
        marker_color=colors,
    ))
    fig2.update_layout(
        title=f"{pair_for_shap}: SHAP Contributions – Latest Observation (top 10)",
        xaxis_title="Feature", yaxis_title="SHAP contribution to predicted TE (%)",
        xaxis_tickangle=-40, template="plotly_white", height=430,
    )
    fig2.show()

In [ ]:
# --- Residual anomaly detection (ResidualAnomalyDetector) ---------------------
# The detector identifies structural changes in (actual - predicted) residuals
# using an ensemble of: MLP autoencoder, Isolation Forest, and regime statistics.
pair_name_anomaly = first_pair

pair_features = rt_features[rt_features["pair"] == pair_name_anomaly].copy()
pair_series_a = prediction_series[prediction_series["pair"] == pair_name_anomaly].copy()

# Align predictions with realized values
pair_features = pair_features.reset_index().rename(columns={"index": "timestamp"})
pair_features["actual_te"] = pair_features["target_te"].where(
    pair_features["target_te"].notna(), pair_features["realized_te"]
)
merged = pair_series_a.merge(
    pair_features[["timestamp", "actual_te"]], on="timestamp", how="left"
).dropna(subset=["predicted_tracking_error", "actual_te"])
merged["residual"] = merged["actual_te"] - merged["predicted_tracking_error"]

short_w = max(10, DEFAULT_WINDOW // 2)
long_w  = max(24, DEFAULT_WINDOW * 2)
min_obs = max(80, long_w * 2)

if len(merged) >= min_obs:
    anomaly_detector = ResidualAnomalyDetector(
        short_window=short_w, long_window=long_w, contamination=0.05, random_state=42
    )
    scored = anomaly_detector.fit_score(
        actual_tracking_error=merged["actual_te"],
        predicted_tracking_error=merged["predicted_tracking_error"],
    )
    latest_anomaly = anomaly_detector.latest_result(
        actual_tracking_error=merged["actual_te"],
        predicted_tracking_error=merged["predicted_tracking_error"],
    )

    flagged = scored[scored["anomaly_detected"] == True]  # noqa: E712
    print(f"Pair            : {pair_name_anomaly}")
    print(f"Observations    : {len(scored)}")
    print(f"Anomalies found : {len(flagged)}")
    print(f"Latest anomaly  : {latest_anomaly['anomaly_detected']}  |  "
          f"type={latest_anomaly['anomaly_type']}  |  confidence={latest_anomaly['confidence']:.2f}")
    print(f"Explanation     : {latest_anomaly['explanation']}")

    # Chart residuals with anomaly overlay
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(
        x=merged["timestamp"], y=merged["residual"] * 100,
        mode="lines", name="Residual (actual − predicted)", line=dict(color="#103a6f", width=2),
    ))
    if not flagged.empty:
        flagged_ts = merged.loc[flagged.index, "timestamp"]
        flagged_res = merged.loc[flagged.index, "residual"] * 100
        fig3.add_trace(go.Scatter(
            x=flagged_ts, y=flagged_res,
            mode="markers", name="Detected Anomaly",
            marker=dict(color="#cf2e2e", size=9, symbol="diamond"),
        ))
    fig3.update_layout(
        title=f"{pair_name_anomaly}: Residuals with Detected Anomalies",
        xaxis_title="Date", yaxis_title="Residual (%)",
        template="plotly_white", height=430,
    )
    fig3.show()
else:
    print(f"Need at least {min_obs} aligned observations (have {len(merged)}). "
          "Run with a longer period to see anomaly results.")